In [1]:
import pandas as pd
import dotenv
dotenv.load_dotenv()

False

In [2]:
df_london_smartmeter = pd.read_csv("/scratch/ejk5818/ts-inverse/data/LondonSmartMeter/london_smart_meters_dataset_without_missing_values_first_30_consumers.csv", index_col='Time', parse_dates=['Time'])
df_kddcup = pd.read_csv("/scratch/ejk5818/ts-inverse/data/KDDCup_2018/kdd_cup_2018_dataset_without_missing_values.csv", index_col='Time', parse_dates=['Time'])
df_electricity_321_hourly = pd.read_csv("/scratch/ejk5818/ts-inverse/data/Electricity321Hourly/electricity_hourly_dataset.csv", index_col='Time', parse_dates=['Time'])
df_electricity_370 = pd.read_csv("/scratch/ejk5818/ts-inverse/data/Electricity370/LD2011_2014_first_40_consumers.csv", index_col='Time', parse_dates=['Time'])
# tno_df.head()
# print(df.columns.tolist()[:1])

In [3]:
import torch
import psutil

# Check if CUDA is available
if torch.cuda.is_available():
    # Get the current device
    torch.set_float32_matmul_precision('high')
    device = torch.cuda.current_device()
    print(f"Using CUDA device: {torch.cuda.get_device_name(device)}")
else:
    device = 'cpu'
    print("CUDA is not available")

print("Physical cores:", psutil.cpu_count(logical=False))
print("Total cores:", psutil.cpu_count(logical=True))
cpu_freq = psutil.cpu_freq()
print(f"Current Frequency: {cpu_freq.current:.2f}Mhz")

# RAM Information
svmem = psutil.virtual_memory()
print(f"Total: {svmem.total / (1024 ** 3):.2f} GB")
print(f"Available: {svmem.available / (1024 ** 3):.2f} GB")
print(f"Used: {svmem.used / (1024 ** 3):.2f} GB")
print(f"Percentage: {svmem.percent}%")

print("Distributed PyTorch available:", torch.distributed.is_available())

/scratch/ejk5818/miniconda3/envs/ts-inverse/lib/python3.11/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Using CUDA device: NVIDIA RTX A6000
Physical cores: 24
Total cores: 48
Current Frequency: 3320.55Mhz
Total: 503.50 GB
Available: 466.04 GB
Used: 37.46 GB
Percentage: 7.4%
Distributed PyTorch available: True


In [4]:
from copy import deepcopy
from ts_inverse.models import FCN_Predictor, CNN_Predictor, GRU_Predictor, JitGRU_Predictor, CNNJitGRU_Predictor, TCN_Predictor, JitSeq2Seq_Predictor
from ts_inverse.utils import grid_search_params
from ts_inverse.workers import AttackTSInverseWorker


def start_multi_process(g_config, a_config, d_config, m_config, pool_size):
    search_args = []
    search_configs = list(grid_search_params(g_config))
    search_attack_configs = list(grid_search_params(a_config))
    search_dataset_settings = list(grid_search_params(d_config))
    search_model_settings = list(grid_search_params(m_config))
    for original_g_config in search_configs:
        for a_config in search_attack_configs:
            g_config = deepcopy(original_g_config)
            g_config.update(a_config)
            for m_config in search_model_settings:
                for d_config in search_dataset_settings:
                    fa_models_config = {
                        'features': [[0]],
                        'input_size': d_config['observation_days'],
                        'output_size': d_config['future_days'],
                    }
                    search_for_all_models_settings = list(grid_search_params(fa_models_config))
                    for fa_models_config in search_for_all_models_settings:
                        g_config['run_number'] = len(search_args)
                        args = (g_config, d_config, m_config, fa_models_config, None)
                        search_args.append(deepcopy(args))

    print(f"Starting {len(search_args)} processes")
    if pool_size == 1:
        for args in search_args:
            AttackTSInverseWorker(args[0]['run_number']).worker_process(*args)


global_config = {
    'logger_service': 'wandb',
    # 'logger_service': 'none',
    'experiment_name': 'ts-inverse_batch1_without_target_reconst_12-6-2024',
    # 'seed': [10, 43, 28, 80, 71], # 28, 80, 71],
    'seed': [43], # 28, 80, 71],
    'batch_size': 1,
    'device': 0,
    'verbose': False,
    'pool_size': 1,
    'run_number': -1,
    'total_variation_alpha_inputs': 0, 
    'total_variation_beta_targets': 0,
    'after_effect': 'none',
    'warmup_number_of_batches': 0,
    'number_of_batches': 1,
    'update_model': False, # Update the model in generating gradients from training data
    'model_evaluation_during_attack': False, # Baselines do not consider this
    'load_lti_model': False,

    'dropout': 0,
    'optimize_dropout': False,
    'dropout_probability_regularizer': 0,
    'dummy_init_method': 'rand',
}

attack_config = [
    {
        'attack_method': 'TS-Inverse',
        # invert attack
        'num_learn_epochs': 0,
        'learn_learning_rate': 1e-3, 
        'attack_batch_size': 32,
        'inversion_batch_size': 1, # global_config['batch_size'],
        'attack_hidden_size': [[768, 512]],
        'quantiles': [[0.1, 0.3, 0.7, 0.9]],
        'attack_loss': ['quantile'],
        'inversion_model': 'ImprovedGradToInputNN_Quantile', 
        'attack_targets': True,
        'learn_optimizer': 'adamW',
        'learn_lr_decay': ['on_plateau'],
        'aux_dataset': None,
        'one_shot_targets': False,

        ## Inversion regularization in optimization attack
        'inversion_regularization_term_inputs': [0], 
        'inversion_regularization_term_targets': [0],
        'inversion_regularization_loss': ['quantile'],

        'lower_res_term': [0],
        'trend_term': [0],
        'trend_loss': ['l1_mean'],
        'trend_reduce_lr': [False],
        'periodicity_term': [0],
        'periodicity_loss': ['l1_mean'],
        'periodicity_reduce_lr': [False],


        ## Optimization attack
        'gradient_loss': ['l1'], 
        'base_num_attack_steps': 500,
        'after_effect': 'clamp_2',
        'optimization_learning_rate': 0.01, 
        'attack_opti_optimizer': ['adam'],
        'attack_opti_lr_decay': ['on_plateau_10'],
        'optimize_dropout': True,
        'clamp_dropout': 1,
        'clamp_dropout_min_max': [[0.0, 1.0]],
        'dropout_probability_regularizer': [1e-5], #1e-6, 1e-7, 1e-8],
        'dropout_mask_init_type': 'halves', #['bernoulli', 'halves', 'uniform', 'p', '1-p'], #['halves', 'bernoulli'],
        'grad_signs_for_inputs': True,
        'grad_signs_for_targets': False,
        'grad_signs_for_dropouts': True, #[False, True],

        'attack_number_of_batches': 1,
    },
]

dataset_config = [
    {
        'dataset': 'electricity_370',
        'columns': [df_electricity_370.columns.tolist()[4:5]],
        'train_stride': 24,
        'validation_stride': 1,
        'observation_days': 1,
        'future_days': 1,
        'normalize': 'minmax',
    },
    {
        'dataset': 'london_smartmeter',
        'columns': [df_london_smartmeter.columns.tolist()[:1]],
        'train_stride': 24, # Is the strides that are attacked
        'validation_stride': 1, # Is the stride that is used for training the invertion model
        'observation_days': 1,
        'future_days': 1,
        'normalize': 'minmax',
    },
    {
        'dataset': 'kddcup',
        'columns': [df_kddcup.columns.tolist()[:1]],
        'train_stride': 24,
        'validation_stride': 1,
        'observation_days': 5,
        'future_days': 2,
        'normalize': 'minmax',
    },    
]

model_config = [
    {
        '_model': FCN_Predictor,
        'hidden_size': 64,
        '_attack_step_multiplier': 10,
    },
    {
        '_model': CNN_Predictor,
        'hidden_size': 64,
        '_attack_step_multiplier': 10,
    },
    {
        '_model': TCN_Predictor,
        'hidden_size': 64,
        'num_levels': 0,
        'kernel_size': 6,
        'dilation_factor': 2,
        'activation': 'relu',
        'use_weight_norm': True,
        'init_weights': True,
        'dropout': 0.1,
        '_attack_step_multiplier': 10,
    },
    # {
    #     '_model': JitGRU_Predictor,
    #     'hidden_size': 64,
    #     '_attack_step_multiplier': 10,
    # },
    # {
    #     '_model': JitSeq2Seq_Predictor,
    #     'hidden_size': 64,
    #     '_attack_step_multiplier': 10,
    # }
][1:2]

start_multi_process(global_config, attack_config, dataset_config, model_config, global_config['pool_size'])

Starting 3 processes
def_c None
Starting attack 0 with config: {'logger_service': 'wandb', 'experiment_name': 'ts-inverse_batch1_without_target_reconst_12-6-2024', 'seed': 43, 'batch_size': 1, 'device': 0, 'verbose': False, 'pool_size': 1, 'run_number': 0, 'total_variation_alpha_inputs': 0, 'total_variation_beta_targets': 0, 'after_effect': 'clamp_2', 'warmup_number_of_batches': 0, 'number_of_batches': 1, 'update_model': False, 'model_evaluation_during_attack': False, 'load_lti_model': False, 'dropout': 0, 'optimize_dropout': True, 'dropout_probability_regularizer': 1e-05, 'dummy_init_method': 'rand', 'attack_method': 'TS-Inverse', 'num_learn_epochs': 0, 'learn_learning_rate': 0.001, 'attack_batch_size': 32, 'inversion_batch_size': 1, 'attack_hidden_size': [768, 512], 'quantiles': [0.1, 0.3, 0.7, 0.9], 'attack_loss': 'quantile', 'inversion_model': 'ImprovedGradToInputNN_Quantile', 'attack_targets': True, 'learn_optimizer': 'adamW', 'learn_lr_decay': 'on_plateau', 'aux_dataset': None, '

wandb: Currently logged in as: ejk5818 (ejk5818-penn-state) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Skipping inversion model training as num_learn_epochs <= 0


custom_step,▁▁▁▁▁▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
grad_diff_loss_mse,█▃▃▃▃▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
inputs/mae/0,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
inputs/mae/mean,▇█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
inputs/mse/0,▇▇█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
inputs/mse/mean,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
inputs/rmse/0,█▅▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
inputs/rmse/mean,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
inputs/smape/0,█▂▂▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
inputs/smape/mean,▇█▅▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+11,...


def_c None
Starting attack 1 with config: {'logger_service': 'wandb', 'experiment_name': 'ts-inverse_batch1_without_target_reconst_12-6-2024', 'seed': 43, 'batch_size': 1, 'device': 0, 'verbose': False, 'pool_size': 1, 'run_number': 1, 'total_variation_alpha_inputs': 0, 'total_variation_beta_targets': 0, 'after_effect': 'clamp_2', 'warmup_number_of_batches': 0, 'number_of_batches': 1, 'update_model': False, 'model_evaluation_during_attack': False, 'load_lti_model': False, 'dropout': 0, 'optimize_dropout': True, 'dropout_probability_regularizer': 1e-05, 'dummy_init_method': 'rand', 'attack_method': 'TS-Inverse', 'num_learn_epochs': 0, 'learn_learning_rate': 0.001, 'attack_batch_size': 32, 'inversion_batch_size': 1, 'attack_hidden_size': [768, 512], 'quantiles': [0.1, 0.3, 0.7, 0.9], 'attack_loss': 'quantile', 'inversion_model': 'ImprovedGradToInputNN_Quantile', 'attack_targets': True, 'learn_optimizer': 'adamW', 'learn_lr_decay': 'on_plateau', 'aux_dataset': None, 'one_shot_targets': Fa

Skipping inversion model training as num_learn_epochs <= 0


KeyboardInterrupt: 

socket.send() raised exception.


Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f4f8a6e1b50>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f513d059650, execution_count=4 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7f51310e0ed0, raw_cell="from copy import deepcopy
from ts_inverse.models i.." transformed_cell="from copy import deepcopy
from ts_inverse.models i.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Bpsu/scratch/ejk5818/ts-inverse/notebooks/05_ts_inverse_reconstruction_attack_batch_size_8.ipynb#W3sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost